In [2]:
import sys
original_sys_path = sys.path.copy()
sys.path.append('../')
from utils_models import *

dq.set_device('gpu')
import warnings
warnings.filterwarnings("ignore")

In [3]:
solver = dq.solver.Tsit5(
                    rtol= 1e-06,
                    atol= 1e-06,
                    safety_factor= 0.9,
                    min_factor= 0.2,
                    max_factor = 5.0,
                    max_steps = int(1e4*1000),
                )


EJ = 2
EC = EJ/4
EL = EJ/20
n_lvls_fluxonium = 5
n_lvls_resonator = 20
Er = 11.46660156/2
qsf = qs.Fluxonium.create(
    n_lvls_fluxonium,
    {"Ej": EJ, "Ec": EC, "El": EL, "phi_ext": 0.0},
    N_pre_diag=120,
    use_linear = False
    )

res = MyResonator.create(
    n_lvls_resonator,
    params = {"ω":Er,"α":0},
    use_linear = False,
)

devices = [qsf, res]
Ns = [device.N for device in devices]
fn = qs.promote(qsf.ops["n"], 0, Ns)
rn = qs.promote(res.ops['n'], 1, Ns)

g_tf = 0.2
system = qs.System.create(devices, couplings=[
    g_tf *  fn @ rn
    ])

system_evals_sorted, system_evecs_sorted, product_indices_sorted_by_eval = calculate_eig(Ns, system.get_H())
driven_op = transform_op_into_dressed_basis_jax(rn, system_evecs_sorted.T)

system_evals_in_product_indices, system_evecs_in_product_indices = system.calculate_eig_linear()
w_d = system_evals_in_product_indices[0,1] - system_evals_in_product_indices[0,0]


In [51]:
dressed_indices_to_keep = []
for ql in range(3):
    for rl in range(70):
        dressed_indices_to_keep.append( find_closest_dressed_index(ql*70+rl, product_indices_sorted_by_eval) )
dressed_indices_to_keep = jnp.array(dressed_indices_to_keep)
def truncate(data: jnp.array):
    return jnp.take(jnp.take(data, dressed_indices_to_keep, axis=0), dressed_indices_to_keep, axis=1)
tot_dim = len(dressed_indices_to_keep)

In [56]:
sorted_dressed_indices_to_keep = sorted(dressed_indices_to_keep)
product_to_truncated_dressed = {}
for ql in range(3):
    for rl in range(70):
        dressed_idx = find_closest_dressed_index(ql*70+rl, product_indices_sorted_by_eval)
        product_to_truncated_dressed[(ql,rl)] = sorted_dressed_indices_to_keep.index(dressed_idx)

In [ ]:
# TODO: implement this later
truncated_dressed_idxes_with_negative_sign = []
for i in range(len(dressed_indices_to_keep)):
    arr = np.array(evecs[i])
    max_abs_index = np.argmax(np.abs(arr))
    max_abs_value = arr[max_abs_index]
    if max_abs_value > 0:
        pass
    elif max_abs_value < 0:
        truncated_dressed_idxes_with_negative_sign.append(i)
        
# Convert dressed_idxes_with_negative_sign to a set for O(1) lookup
dressed_idxes_with_negative_sign_set = set(truncated_dressed_idxes_with_negative_sign)

# Pre-compute the sign multiplier for each dressed index
self.sign_multiplier = {idx: -1 if idx in dressed_idxes_with_negative_sign_set else 1
                for idx in self.filtered_product_to_dressed.values()}

In [52]:
def truncated_dressed_to_product(dressed_dm: qutip.Qobj, 
                        product_to_truncated_dressed: dict, 
                        sign_multiplier:dict,
                        ) -> qutip.Qobj:
    dressed_dm_data = dressed_dm
    if dressed_dm_data.shape[1] == 1:
        dressed_dm_data = qutip.ket2dm(dressed_dm_data)
    dressed_dm_data = dressed_dm_data.full()

    subsystem_dims = [max(indexes) + 1 for indexes in zip(*product_to_truncated_dressed.keys())]
    rho_product = np.zeros((subsystem_dims*2), dtype=complex) # Here rho_product is shaped like (dim1,dim2,dim1,dim2)
    for product_state, dressed_index1 in product_to_truncated_dressed.items():
        for product_state2, dressed_index2 in product_to_truncated_dressed.items():
            element = dressed_dm_data[dressed_index1, dressed_index2] * sign_multiplier[dressed_index1] * sign_multiplier[dressed_index2]
            rho_product[product_state+product_state2] += element # Using index like (lvl1, lvl2, lvl1, lvl2) to access of of the entries

    two_lvl_qbt_dm_size = np.prod(subsystem_dims)
    rho_product = rho_product.reshape((two_lvl_qbt_dm_size,two_lvl_qbt_dm_size))
    rho_product = qutip.Qobj(rho_product, dims=[subsystem_dims, subsystem_dims])

    return rho_product

(210, 210)